# <font color='cyan'>Exploración de Corpus</font>

## Exploración Inicial

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../../reports/bronze_audit.csv')
print(df[df['audit_status'] != 'OK'][['filename','composer','work_title','bwv_number','audit_warnings']].head(20).to_string())

                                                                                     filename composer work_title bwv_number audit_warnings
0                            telemann_kunstderfuge_unknown_tafelmusik_6_(c)icking-archive.mid      NaN        NaN        NaN            NaN
1                 telemann_kunstderfuge_unknown_tafelmusik_5_(trumpets)_(c)icking-archive.mid      NaN        NaN        NaN            NaN
2                      telemann_kunstderfuge_unknown_three-four_suite_3_(c)icking-archive.mid      NaN        NaN        NaN            NaN
3                 telemann_kunstderfuge_unknown_fantasie_per_cembalo_11_(c)icking-archive.mid      NaN        NaN        NaN            NaN
4                 telemann_kunstderfuge_unknown_tafelmusik_1_(trumpets)_(c)icking-archive.mid      NaN        NaN        NaN            NaN
5                             telemann_kunstderfuge_unknown_burlesque_6_(c)icking-archive.mid      NaN        NaN        NaN            NaN
6   telemann_kunstde

In [6]:
print(df['composer'].value_counts().to_string())
print('---')
print(df['extension'].value_counts().to_string())

composer
Bach, Johann Sebastian    770
---
extension
.mid    1593
.krn     770


## Pasos a Seguir en Corpus

In [4]:
df = pd.read_csv('../../reports/enriched_audit.csv')
print('=== Silver Actions ===')
print(df['silver_action'].value_counts().to_string())
print()
print('=== By Source ===')
print(df.groupby('source_id')['silver_action'].value_counts().to_string())

=== Silver Actions ===
silver_action
NEEDS_REVIEW      1316
KEEP_PRIMARY       671
DUPLICATE_DROP     370
KEEP_CONTRAST        3
QUARANTINE           3

=== By Source ===
source_id               silver_action 
bulk_chorales_370       DUPLICATE_DROP    370
bulk_chorales_371       KEEP_PRIMARY      370
kernscores_inventions   KEEP_PRIMARY       30
                        NEEDS_REVIEW       30
kunstderfuge_couperin   NEEDS_REVIEW      118
kunstderfuge_handel     NEEDS_REVIEW      499
kunstderfuge_js         KEEP_PRIMARY      271
kunstderfuge_rameau     NEEDS_REVIEW       24
kunstderfuge_scarlatti  NEEDS_REVIEW      555
kunstderfuge_telemann   NEEDS_REVIEW       60
kunstderfuge_vivaldi    NEEDS_REVIEW       30
                        KEEP_CONTRAST       3
                        QUARANTINE          3


## Check 1

In [5]:
contrast = df[df['source_id'].str.startswith('kunstderfuge_') & 
              (df['composer_resolved'] != 'bach')]
print('=== Contrast files audit_status distribution ===')
print(contrast['audit_status'].value_counts().to_string())

=== Contrast files audit_status distribution ===
audit_status
ERROR      1286
WARNING       6


## Check 2

In [6]:
print()
print('=== Sample NEEDS_REVIEW Handel rows ===')
handel_nr = df[(df['source_id']=='kunstderfuge_handel') & 
               (df['silver_action']=='NEEDS_REVIEW')]
print(handel_nr[['filename','source_id','audit_status','silver_action','enrich_notes']].head(5).to_string())


=== Sample NEEDS_REVIEW Handel rows ===
                                                              filename            source_id audit_status silver_action       enrich_notes
84         handel_kunstderfuge_unknown_hwv-444_1_prelude_(c)yamada.mid  kunstderfuge_handel        ERROR  NEEDS_REVIEW                NaN
85    handel_kunstderfuge_unknown_7_kleine_fugen_(c)icking-archive.mid  kunstderfuge_handel        ERROR  NEEDS_REVIEW  no_catalog_system
86               handel_kunstderfuge_unknown_messiah_48_(c)unknown.mid  kunstderfuge_handel        ERROR  NEEDS_REVIEW  no_catalog_system
87   handel_kunstderfuge_unknown_concerto_grosso_6_3_3_(c)withcomb.mid  kunstderfuge_handel        ERROR  NEEDS_REVIEW  no_catalog_system
88  handel_kunstderfuge_unknown_concerto_grosso_6_10_4_(c)withcomb.mid  kunstderfuge_handel        ERROR  NEEDS_REVIEW  no_catalog_system


## Check 3

In [7]:
print()
print('=== The 3 correct Vivaldi KEEP_CONTRAST ===')
print(df[df['silver_action']=='KEEP_CONTRAST'][['filename','source_id','audit_status']].to_string())


=== The 3 correct Vivaldi KEEP_CONTRAST ===
                                                                            filename             source_id audit_status
2349  vivaldi_kunstderfuge_unknown_concerto_la_stravaganza_4_2_(c)icking-archive.mid  kunstderfuge_vivaldi      WARNING
2354  vivaldi_kunstderfuge_unknown_concerto_la_stravaganza_4_1_(c)icking-archive.mid  kunstderfuge_vivaldi      WARNING
2356  vivaldi_kunstderfuge_unknown_concerto_la_stravaganza_4_3_(c)icking-archive.mid  kunstderfuge_vivaldi      WARNING


## Contar cuantos MIDI son HTML

In [9]:
from pathlib import Path

raw_dir = Path('/Users/auradelagg/aura-lab/bach-propagation/backend/data/raw')
html_count = 0
midi_count = 0

for p in raw_dir.rglob('*.mid'):
    with open(p, 'rb') as f:
        header = f.read(64)
    if b'<!DOCTYPE' in header or b'<html' in header or b'Daily limit' in header:
        html_count += 1
    elif header[:4] == b'MThd':
        midi_count += 1
    else:
        print(f'UNKNOWN: {p.name} | header: {header[:20]}')

print(f'Valid MIDI files : {midi_count}')
print(f'HTML imposters   : {html_count}')

Valid MIDI files : 307
HTML imposters   : 1286


In [13]:
raw_dir = Path('/Users/auradelagg/aura-lab/bach-propagation/backend/data/raw')

valid = []
fake = []

for p in raw_dir.rglob('*.mid'):
    with open(p, 'rb') as f:
        header = f.read(64)
    if header[:4] == b'MThd':
        valid.append(p)
    else:
        fake.append(p)

print(f'=== Valid MIDI by source folder ===')
from collections import Counter
valid_sources = Counter(p.parts[p.parts.index('raw')+1] for p in valid)
for src, count in valid_sources.most_common():
    print(f'  {src:<20} {count}')

print(f'=== Fake (HTML) MIDI by source folder ===')
fake_sources = Counter(p.parts[p.parts.index('raw')+1] for p in fake)
for src, count in fake_sources.most_common():
    print(f'  {src:<20} {count}')

=== Valid MIDI by source folder ===
  bach_js              271
  bach                 30
  vivaldi              6
=== Fake (HTML) MIDI by source folder ===
  scarlatti            555
  handel               499
  couperin             118
  telemann             60
  vivaldi              30
  rameau               24


In [14]:
raw_dir = Path('/Users/auradelagg/aura-lab/bach-propagation/backend/data/raw')
fake = []
for p in raw_dir.rglob('*.mid'):
    with open(p, 'rb') as f:
        header = f.read(64)
    if header[:4] != b'MThd':
        fake.append(p)

print(f'Would delete {len(fake)} files:')
from collections import Counter
sources = Counter(p.parts[p.parts.index('raw')+1] for p in fake)
for src, count in sources.most_common():
    print(f'  {src:<20} {count}')

Would delete 1286 files:
  scarlatti            555
  handel               499
  couperin             118
  telemann             60
  vivaldi              30
  rameau               24


## Nueva Auditoria de Corpus restante

In [3]:
df = pd.read_csv('../../reports/enriched_audit.csv')
print('=== Silver Actions ===')
print(df['silver_action'].value_counts().to_string())
print()
print('=== By Source ===')
print(df.groupby('source_id')['silver_action'].value_counts().to_string())

=== Silver Actions ===
silver_action
KEEP_PRIMARY      671
DUPLICATE_DROP    370
NEEDS_REVIEW       30
KEEP_CONTRAST       3
QUARANTINE          3

=== By Source ===
source_id              silver_action 
bulk_chorales_370      DUPLICATE_DROP    370
bulk_chorales_371      KEEP_PRIMARY      370
kernscores_inventions  KEEP_PRIMARY       30
                       NEEDS_REVIEW       30
kunstderfuge_js        KEEP_PRIMARY      271
kunstderfuge_vivaldi   KEEP_CONTRAST       3
                       QUARANTINE          3


In [4]:
df = pd.read_csv('../../reports/enriched_audit.csv')
keep = df[df['silver_action'] == 'KEEP_PRIMARY']

print('=== Sample slugs by source ===')
for source in keep['source_id'].unique():
    print(f'\n-- {source} --')
    sample = keep[keep['source_id']==source][['filename','silver_slug','catalog_number_resolved','movement_name_decoded']].head(5)
    print(sample.to_string())

print('\n=== Slug collision check ===')
dupes = keep[keep.duplicated('silver_slug', keep=False)]
print(f'Colliding slugs: {len(dupes)}')
if len(dupes) > 0:
    print(dupes[['filename','silver_slug','source_id']].head(10).to_string())

=== Sample slugs by source ===

-- bulk_chorales_371 --
        filename           silver_slug catalog_number_resolved movement_name_decoded
370  chor217.krn  bach_chorale_217.krn                     NaN               Chorale
371  chor203.krn  bach_chorale_203.krn                     NaN               Chorale
372  chor001.krn  bach_chorale_001.krn                     NaN               Chorale
373  chor175.krn  bach_chorale_175.krn                     NaN               Chorale
374  chor015.krn  bach_chorale_015.krn                     NaN               Chorale

-- kunstderfuge_js --
                                filename        silver_slug catalog_number_resolved movement_name_decoded
740    bach_bwv810e_mid_bwv810e_full.mid   bach_BWV810E.mid                 BWV810E                   NaN
741    bach_bwv810d_mid_bwv810d_full.mid   bach_BWV810D.mid                 BWV810D                   NaN
742    bach_bwv806d_mid_bwv806d_full.mid   bach_BWV806D.mid                 BWV806D          

## Plan de Renombramiento

In [6]:
df = pd.read_csv('../../reports/rename_plan.csv')
print('=== Actions ===')
print(df['action'].value_counts().to_string())
print()
print('=== Silver structure ===')
import os
df['silver_subdir'] = df['silver_path'].apply(
    lambda p: '/'.join(p.replace('\\\\','/').split('/')[-3:-1])
)
print(df.groupby('silver_subdir')['action'].count().to_string())
print()
print('=== Collision review ===')
cols = df[df['action']=='COLLISION_RESOLVED'][['source_path','silver_slug','note']]
print(cols.to_string() if len(cols) else 'No collisions.')

=== Actions ===
action
RENAME                672
COLLISION_RESOLVED      2

=== Silver structure ===
silver_subdir
bach/chorales       370
bach/inventions      30
bach/midi           271
contrast/vivaldi      3

=== Collision review ===
                                                                                                                        source_path                             silver_slug                                              note
526  /Users/auradelagg/aura-lab/bach-propagation/backend/data/data/raw/bach_js/bach/1080_c02_mid/bach_1080_c02_mid_unknown_full.mid    bach_BWV1080_02_contrapunctus_02.mid  collision with 1 other file(s) — kept as primary
537  /Users/auradelagg/aura-lab/bach-propagation/backend/data/data/raw/bach_js/bach/1080c02b_mid/bach_1080c02b_mid_unknown_full.mid  bach_BWV1080_02_contrapunctus_02_b.mid                          Art of Fugue variant (b)
